# Usar los modelos finales con tus propios compuestos

**Para quién:** la patrocinadora y su grupo, que lo usan de manera autónoma: el proyecto termina con la entrega y el equipo no se compromete a acompañar la operación ([C.8](../docs/decisiones.md#dc-8)). **Qué hace:** recibe una lista de SMILES y devuelve, para cada compuesto, (1) la probabilidad de que cruce la barrera hematoencefálica, (2) la potencia que el modelo de la fase 2 predice contra cada una de las 26 dianas, y (3) **qué tan cerca queda el compuesto de lo que los modelos vieron al entrenarse**. No hay servicio ni punto de acceso: la reunión del 3 de noviembre decidió que lo que se entrega es este cuaderno.

**Qué no hace, a propósito.** No decide un corte: la probabilidad de la fase 1 se entrega tal cual, porque cortarla en 0.5 cambia de sesgo cuando el compuesto es distinto de lo entrenado ([1.2](../docs/decisiones.md#d1-2)); y no combina las dos fases en un ranking, porque esa combinación sigue pendiente ([C.3](../docs/decisiones.md#dc-3)). Un BBB− **no descarta** un compuesto.

## 1. Qué respalda estas cifras

Los modelos se entrenaron con las filas de **ajuste** (entrenamiento y validación), sin la prueba reservada. Las cifras de desempeño **no se recalculan aquí**: son las de las corridas registradas, y la diferencia entre la validación cruzada y la prueba es la advertencia más importante de este cuaderno.

| fase | modelo | validación cruzada agrupada | prueba reservada (química más nueva) |
|:--|:--|:--|:--|
| 1 · permeabilidad | `svm_rbf` | MCC 0.7017, AUROC 0.919 | MCC 0.339, AUROC 0.793 (1,499 compuestos) |
| 2 · potencia | `aft` censurado | concordancia 0.761 (0.632 en pares que solo tiene NPASS) | concordancia 0.720 (**0.530 en NPASS, 117 pares**) |

**Una cifra de validación cruzada no describe a la química nueva**: con el mismo modelo, exigir que ningún compuesto de entrenamiento se parezca más de 0.4 a uno de validación baja el MCC de la fase 1 en 33 puntos. Por eso el cuaderno calcula la similitud de cada consulta con el entrenamiento y avisa cuando es baja (sección 4). Es una **heurística, no una garantía**: la similitud al vecino más cercano no anticipa bien la dificultad de la prueba reservada.

In [1]:
import sys
from pathlib import Path

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))

import pandas as pd

from src import inferencia

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 20)

modelos = inferencia.cargar()
print(f"Fase 1: {modelos.meta['filas']['fase1']:,} compuestos · {len(modelos.columnas_fase1)} descriptores")
print(f"Fase 2: {modelos.meta['filas']['fase2']:,} pares · {len(modelos.columnas_fase2)} descriptores + {len(modelos.dianas)} dianas")

Fase 1: 6,306 compuestos · 151 descriptores
Fase 2: 56,241 pares · 201 descriptores + 26 dianas


## 2. Tus compuestos

Edita este diccionario: nombre → SMILES. Los seis primeros son **compuestos de la prueba reservada**, que ninguno de los modelos vio al entrenarse: se sortearon con semilla 0 entre los que tienen nombre y alguna potencia medida (`scripts/ranking_por_fase_ejemplo.py`, corrida `2026-11-12-artemio-C.5-ranking-por-fase-ejemplo-prueba`), sin escoger los que mejor lucen. Son seis ejemplos para ver el formato, **no una métrica**. Van además un SMILES inválido y una molécula diminuta para ver cómo se comportan los avisos.

In [2]:
compuestos = {
    "elopiprazole": "Fc1ccc(-c2ccc(CN3CCN(c4cccc5ccoc45)CC3)[nH]2)cc1",
    "pizotifen": "CN1CCC(=C2c3ccccc3CCc3sccc32)CC1",
    "leflunomide": "Cc1oncc1C(=O)Nc1ccc(C(F)(F)F)cc1",
    "droperidol": "O=C(CCCN1CC=C(n2c(=O)[nH]c3ccccc32)CC1)c1ccc(F)cc1",
    "amiodarone": "CCCCc1oc2ccccc2c1C(=O)c1cc(I)c(OCCN(CC)CC)c(I)c1",
    "gluconolactone": "O=C1OC(CO)C(O)C(O)C1O",
    "metano (diminuto)": "C",
    "no es un SMILES": "esto no es un smiles",
}

# Lo medido, solo para los seis de la prueba reservada: la etiqueta BBB de la fase 1 (1 = cruza) y la mayor
# potencia medida en la fase 2 con su diana. Salen de los datos del proyecto (los cuadernos 01 y 02b), no los
# calcula este cuaderno; para tus compuestos no hay equivalente y esas columnas quedan vacías.
medido = {
    "elopiprazole": (1, 9.38, "DRD2"),
    "pizotifen": (1, 8.62, "DRD2"),
    "leflunomide": (0, 4.68, "MAOA"),
    "droperidol": (1, 8.86, "DRD2"),
    "amiodarone": (0, 4.94, "DRD2"),
    "gluconolactone": (0, 4.58, "NFE2L2"),
}

## 3. Predicciones

Una fila por compuesto. Un SMILES que no se puede usar sale con `NaN` y su motivo, y no afecta a los demás.

In [3]:
resultado = inferencia.predecir(list(compuestos.values()), modelos)
resultado.insert(0, "nombre", list(compuestos))
resultado[["nombre", "p_bbb_mas", "mejor_diana", "pActividad_max", "similitud_fase1", "similitud_fase2",
           "aviso_extrapolacion", "motivo"]].round(3)

,nombre,p_bbb_mas,mejor_diana,pActividad_max,similitud_fase1,similitud_fase2,aviso_extrapolacion,motivo
0,elopiprazole,0.949,MAPT,6.407,0.300,0.441,True,
1,pizotifen,0.940,TSPO,7.204,0.545,0.525,False,
2,leflunomide,0.933,IL6,6.482,0.276,0.511,True,
3,droperidol,0.843,TNF,7.268,0.478,0.508,False,
4,amiodarone,0.571,GRIN1,7.105,0.347,0.444,True,
5,gluconolactone,0.275,IL6,4.709,0.344,0.344,True,
6,metano (diminuto),0.915,IL6,4.919,1.000,0.167,True,
7,no es un SMILES,NaN,NaN,NaN,NaN,NaN,False,SMILES ilegible


### Cómo leer la tabla

- `p_bbb_mas`: probabilidad de cruzar la barrera, de 0 a 1. **No es una clase**: 0.5 no es un límite que el modelo garantice.
- `pActividad_<diana>`: potencia predicha (−log₁₀ de la concentración en M; 5 ≈ 10 µM, 6 ≈ 1 µM, 7 ≈ 0.1 µM). **El modelo predice una potencia para cada diana, también las que el compuesto nunca ha tocado**: es una extrapolación de lo aprendido sobre pares medidos, no una medición. `mejor_diana` y `pActividad_max` solo ordenan esas predicciones.
- `similitud_fase1` y `similitud_fase2`: Tanimoto (huellas de Morgan, radio 2) con el compuesto de entrenamiento más parecido. 1.0 significa que **el compuesto estaba en el entrenamiento**, y entonces su predicción no es una prueba del modelo.
- `aviso_extrapolacion`: `True` si alguna similitud es menor de 0.4. Un `False` no garantiza que la predicción sea buena.

## 4. Qué tan cerca queda cada compuesto de lo ya visto

In [4]:
cerca = resultado[["nombre", "similitud_fase1", "similitud_fase2", "aviso_extrapolacion"]].round(3)
cerca.sort_values("similitud_fase1")

,nombre,similitud_fase1,similitud_fase2,aviso_extrapolacion
2,leflunomide,0.276,0.511,True
0,elopiprazole,0.300,0.441,True
5,gluconolactone,0.344,0.344,True
4,amiodarone,0.347,0.444,True
3,droperidol,0.478,0.508,False
1,pizotifen,0.545,0.525,False
6,metano (diminuto),1.000,0.167,True
7,no es un SMILES,NaN,NaN,False


## 5. La potencia contra cada diana

Las diez dianas con mayor potencia predicha para un compuesto, para ver el reparto y no solo el máximo.

In [5]:
def por_diana(nombre: str, n: int = 10) -> pd.Series:
    fila = resultado.loc[resultado["nombre"] == nombre].iloc[0]
    columnas = [f"pActividad_{d}" for d in modelos.dianas]
    s = fila[columnas].astype(float)
    s.index = [c.removeprefix("pActividad_") for c in columnas]
    return s.sort_values(ascending=False).head(n).round(2)


por_diana("droperidol")

TNF       7.27
IL1B      6.95
DRD2      6.70
LRRK2     6.59
KEAP1     6.52
GRIN2B    6.51
PPARG     6.47
IL6       6.47
PTGS2     6.47
NOS2      6.46
Name: 3, dtype: float64

## 6. Un ranking por fase

Dos tablas, **cada fase por separado**: la fase 1 ordena por la probabilidad de cruzar la barrera y la fase 2 por la potencia máxima predicha contra alguna diana. **No hay un ranking combinado**: esa combinación sigue pendiente ([C.3](../docs/decisiones.md#dc-3)). Junto a cada predicción va **lo medido** (`bbb_real` y `pActividad_medida`), para que se vea cuánto se parece una a la otra con compuestos que los modelos no vieron.

Dos advertencias sobre el lugar en la lista. `aviso_extrapolacion` es una bandera **del compuesto**, no de la fase: se enciende si la similitud de cualquiera de las dos queda bajo 0.4, así que una fila puede tener `True` con una similitud alta en la fase que se está leyendo. Un compuesto con el aviso encendido queda lejos de lo que los modelos vieron en alguna fase, y su lugar vale menos. Y en los productos naturales el orden de potencia no está demostrado: la concordancia baja a 0.530 en 117 pares de la prueba reservada, por debajo de una línea base de peso y superficie polar (0.565).

### Fase 1 · permeabilidad

De mayor a menor probabilidad de cruzar la barrera.

In [6]:
validos = resultado[resultado["motivo"] == ""].copy()
validos["bbb_real"] = validos["nombre"].map(lambda n: medido[n][0] if n in medido else None)
validos["pActividad_medida"] = validos["nombre"].map(lambda n: medido[n][1] if n in medido else None)
validos["diana_medida"] = validos["nombre"].map(lambda n: medido[n][2] if n in medido else None)
fase1 = validos.sort_values("p_bbb_mas", ascending=False)[
    ["nombre", "p_bbb_mas", "bbb_real", "similitud_fase1", "aviso_extrapolacion"]
].reset_index(drop=True)
fase1.index += 1
fase1.index.name = "lugar"
fase1.round(3)

,nombre,p_bbb_mas,bbb_real,similitud_fase1,aviso_extrapolacion
lugar,,,,,
1,elopiprazole,0.949,1.0,0.300,True
2,pizotifen,0.940,1.0,0.545,False
3,leflunomide,0.933,0.0,0.276,True
4,metano (diminuto),0.915,NaN,1.000,True
5,droperidol,0.843,1.0,0.478,False
6,amiodarone,0.571,0.0,0.347,True
7,gluconolactone,0.275,0.0,0.344,True


### Fase 2 · potencia

De mayor a menor potencia predicha contra su mejor diana (`pActividad_max`: 5 ≈ 10 µM, 6 ≈ 1 µM).

In [7]:
fase2 = validos.sort_values("pActividad_max", ascending=False)[
    ["nombre", "pActividad_max", "mejor_diana", "pActividad_medida", "diana_medida", "similitud_fase2",
     "aviso_extrapolacion"]
].reset_index(drop=True)
fase2.index += 1
fase2.index.name = "lugar"
fase2.round(3)

,nombre,pActividad_max,mejor_diana,pActividad_medida,diana_medida,similitud_fase2,aviso_extrapolacion
lugar,,,,,,,
1,droperidol,7.268,TNF,8.86,DRD2,0.508,False
2,pizotifen,7.204,TSPO,8.62,DRD2,0.525,False
3,amiodarone,7.105,GRIN1,4.94,DRD2,0.444,True
4,leflunomide,6.482,IL6,4.68,MAOA,0.511,True
5,elopiprazole,6.407,MAPT,9.38,DRD2,0.441,True
6,metano (diminuto),4.919,IL6,NaN,NaN,0.167,True
7,gluconolactone,4.709,IL6,4.58,NFE2L2,0.344,True


## 7. Cómo reentrenar o actualizar

Los modelos viven en `models/finales/` (unos 3 MB) y se regeneran con un comando, que usa las filas de ajuste y **no toca la prueba reservada**:

```bash
./env312/bin/python scripts/entrenar_finales.py
```

El script registra la corrida (parámetros, commit, `sha256` de los datos) en `experimentos/`. Si llegan datos nuevos, **hay que reevaluar con la escalera de semejanza** ([1.2](../docs/decisiones.md#d1-2)) y no con una partición al azar, que infla la cifra. Con otra versión de las librerías las predicciones pueden cambiar en el tercer decimal; `tests/test_inferencia.py` lo detecta contra 25 predicciones de referencia.

**Uso no comercial:** una de las fuentes de la fase 2, NPASS, es CC BY-NC.

# Declaración de uso de IA y contribución del equipo

### Uso de herramientas de IA

| Herramienta utilizada | Propósito de uso | Alcance del apoyo (secciones) | Responsabilidad: qué revisó o validó el equipo |
|---|---|---|---|
| Anthropic. (2026). Claude Code [Agente de programación]. https://claude.com/claude-code | Escribió el código del cuaderno, el módulo de inferencia, el script de entrenamiento final y sus pruebas | Todas las secciones | El equipo decidió el alcance (cuaderno y no servicio, entrenar solo con las filas de ajuste, sin corte ni combinación de fases) y comprobó que los descriptores y el modelo recargado reproducen los de entrenamiento |

### Contribución individual

| Integrante | Contribución al avance |
|---|---|
| Ingrid Pamela Ruiz Puga | Incorporó el cuaderno a la revisión de calidad de la prosa y reforzó su estructura. |
| Artemio Santiago Padilla Robles | Definió el alcance y dirigió el trabajo; la implementación la escribió la herramienta de IA declarada arriba. |
| José Antonio Vázquez Martínez | Revisó el cuaderno. |

### Decisiones principales

1. **Cuaderno, no endpoint.** *Alternativa descartada:* un servicio, porque la patrocinadora pidió lo contrario y no hay quien lo opere.
2. **Sin corte.** *Alternativa descartada:* devolver «BBB+/BBB−» a 0.5, que bajo un cambio de distribución cambia de sesgo.
3. **Aviso de extrapolación por similitud.** *Alternativa descartada:* no avisar. Se asume que es una heurística imperfecta y se dice.